<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# One-cell Conv VAE training on CIFAR-10
# ============================================

!pip install torch torchvision tqdm --quiet

import os
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from tqdm.auto import tqdm
import numpy as np
import random
import matplotlib.pyplot as plt

# --------------------------------------------
# Repro & device
# --------------------------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # typical for speed
    torch.backends.cudnn.benchmark = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --------------------------------------------
# Config
# --------------------------------------------
@dataclass
class VAEConfig:
    # Data
    img_channels: int = 3
    img_size: int = 32
    data_dir: str = "./data"
    batch_size: int = 128
    num_workers: int = 4

    # Model
    latent_dim: int = 128

    # Training
    num_epochs: int = 15
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    beta_kl: float = 1.0
    grad_clip_norm: float = 5.0

    # Logging
    log_interval: int = 100

config = VAEConfig()
print(config)

# --------------------------------------------
# Dataset & Dataloaders (CIFAR-10: real images)
# --------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),  # [0,1]
    transforms.Normalize(mean=(0.5, 0.5, 0.5),
                         std=(0.5, 0.5, 0.5)),  # -> [-1,1]
])

train_dataset = datasets.CIFAR10(
    root=config.data_dir,
    train=True,
    transform=transform,
    download=True,
)

val_dataset = datasets.CIFAR10(
    root=config.data_dir,
    train=False,
    transform=transform,
    download=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

# --------------------------------------------
# Conv VAE (DCGAN-style encoder/decoder)
# --------------------------------------------
class Encoder(nn.Module):
    """
    Encoder:
    3x32x32 -> [Conv blocks] -> feature -> (mu, logvar)
    """
    def __init__(self, img_channels: int, latent_dim: int, img_size: int):
        super().__init__()
        self.conv = nn.Sequential(
            # 3 x 32 x 32
            nn.Conv2d(img_channels, 64, 4, 2, 1),   # 64 x 16 x 16
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),            # 128 x 8 x 8
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),           # 256 x 4 x 4
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1),           # 512 x 2 x 2
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        conv_out_size = img_size // 16  # 32 -> 2
        self.feature_dim = 512 * conv_out_size * conv_out_size

        self.fc_mu = nn.Linear(self.feature_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.feature_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x)                      # [B, 512, 2, 2]
        h = h.view(h.size(0), -1)             # [B, feature_dim]
        mu = self.fc_mu(h)                    # [B, latent_dim]
        logvar = self.fc_logvar(h)            # [B, latent_dim]
        return mu, logvar


class Decoder(nn.Module):
    """
    Decoder:
    z -> [fc] -> 512 x 2 x 2 -> ConvTranspose blocks -> 3x32x32
    """
    def __init__(self, img_channels: int, latent_dim: int, img_size: int):
        super().__init__()
        conv_out_size = img_size // 16  # 2 for 32x32
        self.feature_dim = 512 * conv_out_size * conv_out_size
        self.conv_out_size = conv_out_size

        self.fc = nn.Linear(latent_dim, self.feature_dim)

        self.deconv = nn.Sequential(
            # 512 x 2 x 2
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 256 x 4 x 4
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 128 x 8 x 8
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 64 x 16 x 16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, img_channels, 4, 2, 1),  # 3 x 32 x 32
            nn.Tanh(),  # -> [-1,1]
        )

    def forward(self, z):
        h = self.fc(z)                             # [B, feature_dim]
        h = h.view(h.size(0), 512, self.conv_out_size, self.conv_out_size)
        x_recon = self.deconv(h)
        return x_recon


class ConvVAE(nn.Module):
    def __init__(self, img_channels: int, img_size: int, latent_dim: int):
        super().__init__()
        self.encoder = Encoder(img_channels, latent_dim, img_size)
        self.decoder = Decoder(img_channels, latent_dim, img_size)

    def encode(self, x):
        return self.encoder(x)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


model = ConvVAE(
    img_channels=config.img_channels,
    img_size=config.img_size,
    latent_dim=config.latent_dim,
).to(device)

print(model)

# --------------------------------------------
# VAE loss (MSE + KL)
# --------------------------------------------
def vae_loss_function(recon_x, x, mu, logvar, beta_kl: float = 1.0):
    # MSE over all pixels, summed; KL as usual
    mse = F.mse_loss(recon_x, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total = mse + beta_kl * kl
    return total, mse, kl

# --------------------------------------------
# Optimizer, AMP scaler
# --------------------------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

# --------------------------------------------
# Training / evaluation helpers
# --------------------------------------------
def train_one_epoch(model, dataloader, optimizer, scaler, epoch, config):
    model.train()
    running_total, running_recon, running_kl = 0.0, 0.0, 0.0
    num_samples = len(dataloader.dataset)

    pbar = tqdm(dataloader, desc=f"Train Epoch {epoch}", leave=False)
    for batch_idx, (images, _) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            recon, mu, logvar = model(images)
            total_loss, mse, kl = vae_loss_function(
                recon, images, mu, logvar, beta_kl=config.beta_kl
            )

        scaler.scale(total_loss).backward()

        if config.grad_clip_norm is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip_norm)

        scaler.step(optimizer)
        scaler.update()

        running_total += total_loss.item()
        running_recon += mse.item()
        running_kl += kl.item()

        if (batch_idx + 1) % config.log_interval == 0:
            avg_total = running_total / ((batch_idx + 1) * config.batch_size)
            avg_recon = running_recon / ((batch_idx + 1) * config.batch_size)
            avg_kl = running_kl / ((batch_idx + 1) * config.batch_size)
            pbar.set_postfix(
                loss=f"{avg_total:.3f}",
                recon=f"{avg_recon:.3f}",
                kl=f"{avg_kl:.3f}",
            )

    epoch_total = running_total / num_samples
    epoch_recon = running_recon / num_samples
    epoch_kl = running_kl / num_samples
    return epoch_total, epoch_recon, epoch_kl


@torch.no_grad()
def evaluate(model, dataloader, config):
    model.eval()
    running_total, running_recon, running_kl = 0.0, 0.0, 0.0
    num_samples = len(dataloader.dataset)

    for images, _ in dataloader:
        images = images.to(device, non_blocking=True)
        recon, mu, logvar = model(images)
        total_loss, mse, kl = vae_loss_function(
            recon, images, mu, logvar, beta_kl=config.beta_kl
        )
        running_total += total_loss.item()
        running_recon += mse.item()
        running_kl += kl.item()

    epoch_total = running_total / num_samples
    epoch_recon = running_recon / num_samples
    epoch_kl = running_kl / num_samples
    return epoch_total, epoch_recon, epoch_kl

# --------------------------------------------
# Helper: denormalize and show 16 generated images
# --------------------------------------------
@torch.no_grad()
def show_samples(model, config, num_samples: int = 16):
    model.eval()
    z = torch.randn(num_samples, config.latent_dim, device=device)
    samples = model.decode(z)

    # denorm from [-1,1] -> [0,1]
    samples = (samples * 0.5 + 0.5).clamp(0, 1).cpu()

    grid = make_grid(samples, nrow=4)
    np_img = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(4, 4))
    plt.imshow(np_img)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# --------------------------------------------
# Main training loop
# --------------------------------------------
best_val = float("inf")

for epoch in range(1, config.num_epochs + 1):
    train_total, train_recon, train_kl = train_one_epoch(
        model, train_loader, optimizer, scaler, epoch, config
    )
    val_total, val_recon, val_kl = evaluate(model, val_loader, config)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_total:.4f} (recon={train_recon:.4f}, kl={train_kl:.4f}) | "
        f"val_loss={val_total:.4f} (recon={val_recon:.4f}, kl={val_kl:.4f})"
    )

    # After each epoch: show 16 generated images
    show_samples(model, config, num_samples=16)

    if val_total < best_val:
        best_val = val_total
        torch.save(model.state_dict(), "conv_vae_cifar10_best.pth")
        print("Saved best model.")
